# 🌍 Project 32 — Water Frequency Map

## 1. Introduce the Project

### Goal

In Project 31, we calculated **how much water area** a lake had each month.

Now we go one step further:

> **How often is each individual pixel covered by water?**

Instead of one water-area number per month, we will create a **water frequency map**.

For example:

```text
Pixel A → water in 10/12 months → 83%
Pixel B → water in 6/12 months  → 50%
Pixel C → water in 1/12 months  → 8%
```

This helps us distinguish:

* **Permanent water** → water almost all the time
* **Seasonal water** → water during some months
* **Rare/infrequent water** → water only occasionally

---

# 2. Big Picture

```text
12 Monthly Images
       ↓
Calculate NDWI
       ↓
Water Mask for Each Month
       ↓
12 Water Masks
       ↓
Add Them Together
       ↓
Number of Times Each Pixel = Water
       ↓
Divide by 12
       ↓
Water Frequency %
       ↓
Frequency Map
```

The key idea is:

```text
Water Frequency =
Number of water observations
-----------------------------
Total observations
```

For 12 months:

```text
10 water months / 12 months
= 83.3%
```

---

# 3. Main Concept — Pixel Frequency

This is the most important new concept in this lesson.

Imagine one pixel observed for 5 months:

```text
January    → Water = 1
February   → Water = 1
March      → Water = 0
April      → Water = 1
May        → Water = 0
```

Add them:

```text
1 + 1 + 0 + 1 + 0 = 3
```

So this pixel was water:

```text
3 out of 5 months
```

Frequency:

```text
3 / 5 = 0.60
```

or:

```text
60%
```

---

## Why is this useful?

A normal water map answers:

> Where is water **now**?

A water frequency map answers:

> Where is water **repeatedly present over time**?

This is much more useful for long-term analysis.

---

# 4. Connect to Previous Lessons

This project combines:

### Project 9

NDWI

```text
Green + NIR
     ↓
Water detection
```

### Project 31

Monthly water monitoring

```text
Month 1
Month 2
...
Month 12
```

### New concept

Combine the monthly water masks:

```text
Water Mask 1
+
Water Mask 2
+
...
+
Water Mask 12
```

to calculate frequency.

---

# 5. Step 1 — Define the Study Area

Use the same type of lake boundary from Project 31.

```javascript
var studyArea = geometry;
```

The important part is that the same study area is used throughout the analysis.

---

# 6. Step 2 — Load Sentinel-2

We use our standard Sentinel-2 dataset:

```javascript
var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(studyArea)
.filterDate("2025-01-01", "2026-01-01");
```

This gives us all Sentinel-2 images for one year.

---

# 7. Step 3 — Cloud Masking

We need relatively clean observations.

```javascript
function maskS2(image) {

  var scl = image.select("SCL");

  var mask = scl.neq(3)
    .and(scl.neq(8))
    .and(scl.neq(9))
    .and(scl.neq(10));

  return image.updateMask(mask);
}

var cleanCollection = collection.map(maskS2);
```

The important idea:

```text
Sentinel-2 Images
       ↓
Remove unwanted pixels
       ↓
Cleaner Collection
```

---

# 8. Step 4 — Create the Monthly Water Masks

We already know how to create a monthly composite.

First:

```javascript
var months = ee.List.sequence(1, 12);
```

This gives:

```text
1, 2, 3, ... 12
```

Now create a function:

```javascript
function monthlyWater(month) {

  var start = ee.Date.fromYMD(2025, month, 1);

  var end = start.advance(1, "month");

  var image = cleanCollection
    .filterDate(start, end)
    .median()
    .clip(studyArea);

  var ndwi = image
    .normalizedDifference(["B3", "B8"]);

  return ndwi.gt(0.2)
    .rename("water");
}
```

### What does this function do?

For **one month**:

```text
Filter month
    ↓
Median composite
    ↓
NDWI
    ↓
Water threshold
    ↓
Water mask
```

The output is an image where:

```text
1 = Water
0 = Not water
```

---

# 9. Step 5 — Create 12 Water Masks

Now we apply the function to all months.

```javascript
var monthlyWaterMasks = ee.ImageCollection.fromImages(
  months.map(monthlyWater)
);
```

This creates an `ImageCollection` containing:

```text
January water mask
February water mask
March water mask
...
December water mask
```

So now we have:

```text
12 Images
```

where every image contains:

```text
1 = Water
0 = Not water
```

---

# 10. Step 6 — Count Water Observations

This is the most important step.

We use:

```javascript
var waterCount = monthlyWaterMasks.sum();
```

### What does `.sum()` do?

It adds the pixel values across all 12 images.

For example:

```text
January    1
February   1
March      0
April      1
May        1
...
```

The pixels are added **individually**.

So the output could be:

```text
Pixel A → 12
Pixel B → 7
Pixel C → 2
```

Meaning:

```text
Pixel A → water 12 months
Pixel B → water 7 months
Pixel C → water 2 months
```

This is our **water occurrence count**.

---

# 11. Step 7 — Calculate Water Frequency

Now divide by the number of observations.

```javascript
var waterFrequency = waterCount.divide(12);
```

Now the values range approximately from:

```text
0 → 1
```

For example:

```text
0.00 → 0% water frequency
0.25 → 25%
0.50 → 50%
0.75 → 75%
1.00 → 100%
```

If you want the result directly as a percentage:

```javascript
var waterFrequencyPercent =
  waterFrequency.multiply(100);
```

---

# 12. Step 8 — Visualize the Frequency Map

```javascript
Map.centerObject(studyArea, 10);

Map.addLayer(
  waterFrequencyPercent,
  {
    min: 0,
    max: 100
  },
  "Water Frequency (%)"
);
```

The map now shows the frequency of water occurrence.

Conceptually:

```text
100%
 ↓
Water almost every month

50%
 ↓
Water about half the year

0%
 ↓
Rarely or never water
```

---

# 13. Understanding the Map

Imagine your lake has this pattern:

```text
             Lake Center
        ┌─────────────────┐
        │     100%        │
        │   Permanent     │
        │     water       │
        │                 │
        │  70%       60%  │
        │ Seasonal water  │
        └─────────────────┘
              Shore
```

The center of a lake will often have a higher frequency.

The shoreline may have lower frequency because water expands and contracts seasonally.

This makes the frequency map much more informative than a single water map.

---

# 14. Complete Workflow

```text
Sentinel-2
    ↓
12 Monthly Composites
    ↓
NDWI
    ↓
12 Water Masks
    ↓
Sum Pixel Values
    ↓
Water Occurrence Count
    ↓
Divide by 12
    ↓
Water Frequency
    ↓
Frequency Map
```

---

# Complete Code

```javascript
// ========================================
// 1. Study Area
// ========================================

var studyArea = geometry;


// ========================================
// 2. Sentinel-2 Collection
// ========================================

var collection = ee.ImageCollection(
  "COPERNICUS/S2_SR_HARMONIZED"
)
.filterBounds(studyArea)
.filterDate("2025-01-01", "2026-01-01");


// ========================================
// 3. Cloud Masking
// ========================================

function maskS2(image) {

  var scl = image.select("SCL");

  var mask = scl.neq(3)
    .and(scl.neq(8))
    .and(scl.neq(9))
    .and(scl.neq(10));

  return image.updateMask(mask);
}


var cleanCollection = collection.map(maskS2);


// ========================================
// 4. Create Months
// ========================================

var months = ee.List.sequence(1, 12);


// ========================================
// 5. Create Monthly Water Mask
// ========================================

function monthlyWater(month) {

  var start = ee.Date.fromYMD(
    2025,
    month,
    1
  );

  var end = start.advance(
    1,
    "month"
  );


  var image = cleanCollection
    .filterDate(start, end)
    .median()
    .clip(studyArea);


  // Calculate NDWI

  var ndwi = image
    .normalizedDifference(["B3", "B8"]);


  // Water threshold

  var water = ndwi
    .gt(0.2)
    .rename("water");


  return water;
}


// ========================================
// 6. Create 12 Monthly Water Masks
// ========================================

var monthlyWaterMasks =
  ee.ImageCollection.fromImages(
    months.map(monthlyWater)
  );


// ========================================
// 7. Count Water Observations
// ========================================

var waterCount =
  monthlyWaterMasks.sum();


// ========================================
// 8. Calculate Water Frequency
// ========================================

var waterFrequency =
  waterCount.divide(12);


// Convert to percentage

var waterFrequencyPercent =
  waterFrequency.multiply(100);


// ========================================
// 9. Display Map
// ========================================

Map.centerObject(
  studyArea,
  10
);

Map.addLayer(
  waterFrequencyPercent,
  {
    min: 0,
    max: 100
  },
  "Water Frequency (%)"
);
```

---

# Things to Remember

### 1. Water frequency is a long-term measure

It does not tell you:

> Is this pixel water today?

It tells you:

> How often was this pixel classified as water during the observation period?

---

### 2. `.sum()` is the key new operation

If each image contains:

```text
1 = Water
0 = Not water
```

then:

```javascript
monthlyWaterMasks.sum()
```

counts how many times each pixel was water.

---

### 3. Frequency needs a denominator

We calculate:

```text
Water Frequency
=
Water Observations
÷
Total Observations
```

For 12 valid monthly observations:

```text
Water Count ÷ 12
```

---

### 4. Frequency is different from water area

**Water area:**

```text
How large is the water body?
```

**Water frequency:**

```text
How consistently is each location covered by water?
```

---

# Common Beginner Mistakes

## ❌ Mistake 1 — Using `.mean()` instead of `.sum()`

We want to count:

```text
How many times = Water?
```

Therefore:

```javascript
.sum()
```

is appropriate.

---

## ❌ Mistake 2 — Forgetting to divide by the number of observations

If:

```text
waterCount = 9
```

that means:

> Water was detected 9 times.

It does **not** directly mean 90%.

We need:

```text
9 / 12 = 75%
```

---

## ❌ Mistake 3 — Assuming 12 months always means 12 valid observations

Some months may have:

* too much cloud
* no usable imagery
* poor-quality observations

So in research, you should consider the **number of valid observations**, not blindly assume 12.

---

## ❌ Mistake 4 — Thinking high frequency automatically means permanent water

For example:

```text
90% frequency
```

means water was detected very frequently during your analysis period.

It does not necessarily prove that the water body is permanently present year-round.

---

# Your Practice — Project 32.1

Create a **Water Frequency Map** for a lake or reservoir.

### Requirements

1. Use Sentinel-2:

```javascript
"COPERNICUS/S2_SR_HARMONIZED"
```

2. Use one year of data.
3. Create monthly composites.
4. Calculate NDWI.
5. Create monthly water masks.
6. Combine the 12 masks.
7. Calculate water occurrence.
8. Calculate water frequency.
9. Display the final frequency map.

### Your final output should show:

```text
0% ─────────────── 100%
Rare              Frequent
water             water
```

---

# Conceptual Questions

### 1.

If a pixel has this record:

```text
1, 1, 1, 0, 0, 1
```

How many times was it water?

What is its water frequency?

---

### 2.

Why do we use `.sum()` after creating the monthly **0/1 water masks**?

---

### 3.

What is the difference between a **water frequency map** and the monthly water-area chart from Project 31?
